In [ ]:
import gc
import math
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
PARTITION_SEED = 42
TEST_SIZE = 0.3
PARTITION_ROOT = Path('/content/drive/MyDrive/Array_submission/Prepared_partitions/splits')
PARTITION_ROOT.mkdir(parents=True, exist_ok=True)
KDD_SOURCES = ['cic2017', 'cic2018', 'darpa', 'unsw-nb15']
CIC_SOURCES = ['CIC-IDS2017', 'CSE-CIC-IDS2018', 'DARPA', 'UNSW-NB15']


In [3]:
def load_kdd_nsl_pool():
    darpa = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/Darpa/Darpa_Pcap_KDD_features.csv', low_memory=False)
    unsw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/UNSW-NB15/UNSW_Pcap_NSL_features.csv', low_memory=False)
    cic17 = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2017/CIC_Pcap_NSL_features.csv', low_memory=False)
    cic18 = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2018/CIC18_Pcap_KDD_features.csv', low_memory=False)
    cic17 = cic17.rename(columns={'category': 'label'})
    unsw = unsw.drop(columns=['label'], errors='ignore').rename(columns={'attack_type': 'label'})
    darpa = darpa.drop(columns=['label', 'difficulty_level'], errors='ignore').rename(columns={'attack_family': 'label'})
    for frame, source_name in [(darpa, 'darpa'), (unsw, 'unsw-nb15'), (cic17, 'cic2017'), (cic18, 'cic2018')]:
        frame['source_pcap'] = source_name
    frame = pd.concat([darpa, unsw, cic17, cic18], ignore_index=True)
    frame['original_label'] = frame['label'].copy()
    labels = frame['label'].astype('string').str.strip().str.replace('-\\s*attempted\\s*$', '', regex=True, flags=re.IGNORECASE).str.strip()
    normalized = labels.str.casefold().str.replace('[_-]+', ' ', regex=True).str.replace('\\s+', ' ', regex=True).str.strip()
    frame['label'] = labels
    frame.loc[normalized.isin(['normal', 'benign']), 'label'] = 'benign'
    frame.loc[normalized.isin(['probe', 'reconnaissance']), 'label'] = 'reconnaissance'
    frame.loc[normalized.str.match('^dos(?:\\s|$)', na=False), 'label'] = 'dos'
    frame.loc[normalized.isin(['backdoor', 'backdoors']), 'label'] = 'backdoor'
    frame.loc[normalized.eq('r2l'), 'label'] = 'r2l'
    frame.loc[normalized.eq('u2r'), 'label'] = 'u2r'
    frame.loc[normalized.str.startswith('web attack', na=False), 'label'] = 'web attack'
    frame.loc[normalized.eq('infiltration'), 'label'] = 'infiltration'
    frame['label'] = frame['label'].astype('string').str.casefold()
    frame['binary_label'] = np.where(frame['label'].eq('benign'), 'benign', 'attack')
    frame['feature_schema'] = 'kdd_nsl'
    return frame.loc[~frame['label'].eq('heartbleed')].reset_index(drop=True)


In [4]:
def load_unsw_pool():
    darpa = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/Darpa/Darpa_Pcap_UNSW_features.csv', low_memory=False)
    unsw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/UNSW-NB15/UNSW_Pcap_UNSW_features.csv', low_memory=False)
    cic17 = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/CIC2017/CIC_Pcap_UNSW_features.csv', low_memory=False)
    cic18 = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/CIC2018/CIC18_Pcap_UNSW_features.csv', low_memory=False)
    cic17 = cic17.drop(columns=['label'], errors='ignore').rename(columns={'category': 'label'})
    unsw = unsw.drop(columns=['label'], errors='ignore').rename(columns={'category': 'label'})
    cic18 = cic18.rename(columns={'Label': 'label'})
    darpa = darpa.drop(columns=['label'], errors='ignore').rename(columns={'attack_cat': 'label'})
    for frame, source_name in [(darpa, 'darpa'), (unsw, 'unsw-nb15'), (cic17, 'cic2017'), (cic18, 'cic2018')]:
        frame['source_pcap'] = source_name
    frame = pd.concat([darpa, unsw, cic17, cic18], ignore_index=True)
    frame['original_label'] = frame['label'].copy()
    labels = frame['label'].astype('string').str.strip().str.replace('(?:\\s*[-_]\\s*|\\s+)attempted\\s*$', '', regex=True, flags=re.IGNORECASE).str.strip()
    normalized = labels.str.casefold().str.replace('[_-]+', ' ', regex=True).str.replace('\\s+', ' ', regex=True).str.strip()
    frame['label'] = labels.str.casefold()
    frame.loc[normalized.isin(['normal', 'benign']), 'label'] = 'benign'
    frame.loc[normalized.isin(['probe', 'reconnaissance']), 'label'] = 'reconnaissance'
    frame.loc[normalized.str.match('^dos(?:\\s|$)', na=False), 'label'] = 'dos'
    frame.loc[normalized.isin(['backdoor', 'backdoors']), 'label'] = 'backdoor'
    frame.loc[normalized.eq('r2l'), 'label'] = 'r2l'
    frame.loc[normalized.eq('u2r'), 'label'] = 'u2r'
    frame.loc[normalized.str.startswith('web attack', na=False), 'label'] = 'web attack'
    frame.loc[normalized.eq('infiltration'), 'label'] = 'infiltration'
    frame['binary_label'] = np.where(frame['label'].eq('benign'), 'benign', 'attack')
    frame['feature_schema'] = 'unsw_nb15'
    return frame.loc[~frame['label'].eq('heartbleed')].reset_index(drop=True)


In [5]:
CIC2017_FEATURE_COLUMNS = ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
CICFLOWMETER_TO_CIC2017 = {'Dst Port': 'Destination Port', 'Total Fwd Packet': 'Total Fwd Packets', 'Total Bwd packets': 'Total Backward Packets', 'Total Length of Fwd Packet': 'Total Length of Fwd Packets', 'Total Length of Bwd Packet': 'Total Length of Bwd Packets', 'Packet Length Min': 'Min Packet Length', 'Packet Length Max': 'Max Packet Length', 'CWR Flag Count': 'CWE Flag Count', 'Fwd Segment Size Avg': 'Avg Fwd Segment Size', 'Bwd Segment Size Avg': 'Avg Bwd Segment Size', 'Fwd Bytes/Bulk Avg': 'Fwd Avg Bytes/Bulk', 'Fwd Packet/Bulk Avg': 'Fwd Avg Packets/Bulk', 'Fwd Bulk Rate Avg': 'Fwd Avg Bulk Rate', 'Bwd Bytes/Bulk Avg': 'Bwd Avg Bytes/Bulk', 'Bwd Packet/Bulk Avg': 'Bwd Avg Packets/Bulk', 'Bwd Bulk Rate Avg': 'Bwd Avg Bulk Rate', 'FWD Init Win Bytes': 'Init_Win_bytes_forward', 'Bwd Init Win Bytes': 'Init_Win_bytes_backward', 'Fwd Act Data Pkts': 'act_data_pkt_fwd', 'Fwd Seg Size Min': 'min_seg_size_forward'}
UNSW_TO_CIC2017 = {'dst_port': 'Destination Port', 'flow_duration': 'Flow Duration', 'tot_fwd_pkts': 'Total Fwd Packets', 'tot_bwd_pkts': 'Total Backward Packets', 'totlen_fwd_pkts': 'Total Length of Fwd Packets', 'totlen_bwd_pkts': 'Total Length of Bwd Packets', 'fwd_pkt_len_max': 'Fwd Packet Length Max', 'fwd_pkt_len_min': 'Fwd Packet Length Min', 'fwd_pkt_len_mean': 'Fwd Packet Length Mean', 'fwd_pkt_len_std': 'Fwd Packet Length Std', 'bwd_pkt_len_max': 'Bwd Packet Length Max', 'bwd_pkt_len_min': 'Bwd Packet Length Min', 'bwd_pkt_len_mean': 'Bwd Packet Length Mean', 'bwd_pkt_len_std': 'Bwd Packet Length Std', 'flow_byts_s': 'Flow Bytes/s', 'flow_pkts_s': 'Flow Packets/s', 'flow_iat_mean': 'Flow IAT Mean', 'flow_iat_std': 'Flow IAT Std', 'flow_iat_max': 'Flow IAT Max', 'flow_iat_min': 'Flow IAT Min', 'fwd_iat_tot': 'Fwd IAT Total', 'fwd_iat_mean': 'Fwd IAT Mean', 'fwd_iat_std': 'Fwd IAT Std', 'fwd_iat_max': 'Fwd IAT Max', 'fwd_iat_min': 'Fwd IAT Min', 'bwd_iat_tot': 'Bwd IAT Total', 'bwd_iat_mean': 'Bwd IAT Mean', 'bwd_iat_std': 'Bwd IAT Std', 'bwd_iat_max': 'Bwd IAT Max', 'bwd_iat_min': 'Bwd IAT Min', 'fwd_psh_flags': 'Fwd PSH Flags', 'bwd_psh_flags': 'Bwd PSH Flags', 'fwd_urg_flags': 'Fwd URG Flags', 'bwd_urg_flags': 'Bwd URG Flags', 'fwd_header_len': 'Fwd Header Length', 'bwd_header_len': 'Bwd Header Length', 'fwd_pkts_s': 'Fwd Packets/s', 'bwd_pkts_s': 'Bwd Packets/s', 'pkt_len_min': 'Min Packet Length', 'pkt_len_max': 'Max Packet Length', 'pkt_len_mean': 'Packet Length Mean', 'pkt_len_std': 'Packet Length Std', 'pkt_len_var': 'Packet Length Variance', 'fin_flag_cnt': 'FIN Flag Count', 'syn_flag_cnt': 'SYN Flag Count', 'rst_flag_cnt': 'RST Flag Count', 'psh_flag_cnt': 'PSH Flag Count', 'ack_flag_cnt': 'ACK Flag Count', 'urg_flag_cnt': 'URG Flag Count', 'cwr_flag_count': 'CWE Flag Count', 'ece_flag_cnt': 'ECE Flag Count', 'down_up_ratio': 'Down/Up Ratio', 'pkt_size_avg': 'Average Packet Size', 'fwd_seg_size_avg': 'Avg Fwd Segment Size', 'bwd_seg_size_avg': 'Avg Bwd Segment Size', 'fwd_byts_b_avg': 'Fwd Avg Bytes/Bulk', 'fwd_pkts_b_avg': 'Fwd Avg Packets/Bulk', 'fwd_blk_rate_avg': 'Fwd Avg Bulk Rate', 'bwd_byts_b_avg': 'Bwd Avg Bytes/Bulk', 'bwd_pkts_b_avg': 'Bwd Avg Packets/Bulk', 'bwd_blk_rate_avg': 'Bwd Avg Bulk Rate', 'subflow_fwd_pkts': 'Subflow Fwd Packets', 'subflow_fwd_byts': 'Subflow Fwd Bytes', 'subflow_bwd_pkts': 'Subflow Bwd Packets', 'subflow_bwd_byts': 'Subflow Bwd Bytes', 'init_fwd_win_byts': 'Init_Win_bytes_forward', 'init_bwd_win_byts': 'Init_Win_bytes_backward', 'fwd_act_data_pkts': 'act_data_pkt_fwd', 'fwd_seg_size_min': 'min_seg_size_forward', 'active_mean': 'Active Mean', 'active_std': 'Active Std', 'active_max': 'Active Max', 'active_min': 'Active Min', 'idle_mean': 'Idle Mean', 'idle_std': 'Idle Std', 'idle_max': 'Idle Max', 'idle_min': 'Idle Min'}
CSE2018_FEATURE_COLUMNS = ['Dst Port', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWR Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
CIC_TO_CSE2018_ALIASES = {'Destination Port': 'Dst Port', 'dst_port': 'Dst Port', 'protocol': 'Protocol', 'flow_duration': 'Flow Duration', 'Total Fwd Packets': 'Tot Fwd Pkts', 'Total Fwd Packet': 'Tot Fwd Pkts', 'tot_fwd_pkts': 'Tot Fwd Pkts', 'Total Backward Packets': 'Tot Bwd Pkts', 'Total Bwd packets': 'Tot Bwd Pkts', 'tot_bwd_pkts': 'Tot Bwd Pkts', 'Total Length of Fwd Packets': 'TotLen Fwd Pkts', 'Total Length of Fwd Packet': 'TotLen Fwd Pkts', 'totlen_fwd_pkts': 'TotLen Fwd Pkts', 'Total Length of Bwd Packets': 'TotLen Bwd Pkts', 'Total Length of Bwd Packet': 'TotLen Bwd Pkts', 'totlen_bwd_pkts': 'TotLen Bwd Pkts', 'Fwd Packet Length Max': 'Fwd Pkt Len Max', 'fwd_pkt_len_max': 'Fwd Pkt Len Max', 'Fwd Packet Length Min': 'Fwd Pkt Len Min', 'fwd_pkt_len_min': 'Fwd Pkt Len Min', 'Fwd Packet Length Mean': 'Fwd Pkt Len Mean', 'fwd_pkt_len_mean': 'Fwd Pkt Len Mean', 'Fwd Packet Length Std': 'Fwd Pkt Len Std', 'fwd_pkt_len_std': 'Fwd Pkt Len Std', 'Bwd Packet Length Max': 'Bwd Pkt Len Max', 'bwd_pkt_len_max': 'Bwd Pkt Len Max', 'Bwd Packet Length Min': 'Bwd Pkt Len Min', 'bwd_pkt_len_min': 'Bwd Pkt Len Min', 'Bwd Packet Length Mean': 'Bwd Pkt Len Mean', 'bwd_pkt_len_mean': 'Bwd Pkt Len Mean', 'Bwd Packet Length Std': 'Bwd Pkt Len Std', 'bwd_pkt_len_std': 'Bwd Pkt Len Std', 'Flow Bytes/s': 'Flow Byts/s', 'flow_byts_s': 'Flow Byts/s', 'Flow Packets/s': 'Flow Pkts/s', 'flow_pkts_s': 'Flow Pkts/s', 'flow_iat_mean': 'Flow IAT Mean', 'flow_iat_std': 'Flow IAT Std', 'flow_iat_max': 'Flow IAT Max', 'flow_iat_min': 'Flow IAT Min', 'Fwd IAT Total': 'Fwd IAT Tot', 'fwd_iat_tot': 'Fwd IAT Tot', 'fwd_iat_mean': 'Fwd IAT Mean', 'fwd_iat_std': 'Fwd IAT Std', 'fwd_iat_max': 'Fwd IAT Max', 'fwd_iat_min': 'Fwd IAT Min', 'Bwd IAT Total': 'Bwd IAT Tot', 'bwd_iat_tot': 'Bwd IAT Tot', 'bwd_iat_mean': 'Bwd IAT Mean', 'bwd_iat_std': 'Bwd IAT Std', 'bwd_iat_max': 'Bwd IAT Max', 'bwd_iat_min': 'Bwd IAT Min', 'fwd_psh_flags': 'Fwd PSH Flags', 'bwd_psh_flags': 'Bwd PSH Flags', 'fwd_urg_flags': 'Fwd URG Flags', 'bwd_urg_flags': 'Bwd URG Flags', 'Fwd Header Length': 'Fwd Header Len', 'fwd_header_len': 'Fwd Header Len', 'Bwd Header Length': 'Bwd Header Len', 'bwd_header_len': 'Bwd Header Len', 'Fwd Packets/s': 'Fwd Pkts/s', 'fwd_pkts_s': 'Fwd Pkts/s', 'Bwd Packets/s': 'Bwd Pkts/s', 'bwd_pkts_s': 'Bwd Pkts/s', 'Min Packet Length': 'Pkt Len Min', 'Packet Length Min': 'Pkt Len Min', 'pkt_len_min': 'Pkt Len Min', 'Max Packet Length': 'Pkt Len Max', 'Packet Length Max': 'Pkt Len Max', 'pkt_len_max': 'Pkt Len Max', 'Packet Length Mean': 'Pkt Len Mean', 'pkt_len_mean': 'Pkt Len Mean', 'Packet Length Std': 'Pkt Len Std', 'pkt_len_std': 'Pkt Len Std', 'Packet Length Variance': 'Pkt Len Var', 'pkt_len_var': 'Pkt Len Var', 'FIN Flag Count': 'FIN Flag Cnt', 'fin_flag_cnt': 'FIN Flag Cnt', 'SYN Flag Count': 'SYN Flag Cnt', 'syn_flag_cnt': 'SYN Flag Cnt', 'RST Flag Count': 'RST Flag Cnt', 'rst_flag_cnt': 'RST Flag Cnt', 'PSH Flag Count': 'PSH Flag Cnt', 'psh_flag_cnt': 'PSH Flag Cnt', 'ACK Flag Count': 'ACK Flag Cnt', 'ack_flag_cnt': 'ACK Flag Cnt', 'URG Flag Count': 'URG Flag Cnt', 'urg_flag_cnt': 'URG Flag Cnt', 'CWE Flag Count': 'CWR Flag Count', 'cwr_flag_count': 'CWR Flag Count', 'ECE Flag Count': 'ECE Flag Cnt', 'ece_flag_cnt': 'ECE Flag Cnt', 'down_up_ratio': 'Down/Up Ratio', 'Average Packet Size': 'Pkt Size Avg', 'pkt_size_avg': 'Pkt Size Avg', 'Avg Fwd Segment Size': 'Fwd Seg Size Avg', 'fwd_seg_size_avg': 'Fwd Seg Size Avg', 'Avg Bwd Segment Size': 'Bwd Seg Size Avg', 'bwd_seg_size_avg': 'Bwd Seg Size Avg', 'Fwd Avg Bytes/Bulk': 'Fwd Byts/b Avg', 'Fwd Bytes/Bulk Avg': 'Fwd Byts/b Avg', 'fwd_byts_b_avg': 'Fwd Byts/b Avg', 'Fwd Avg Packets/Bulk': 'Fwd Pkts/b Avg', 'Fwd Packet/Bulk Avg': 'Fwd Pkts/b Avg', 'fwd_pkts_b_avg': 'Fwd Pkts/b Avg', 'Fwd Avg Bulk Rate': 'Fwd Blk Rate Avg', 'Fwd Bulk Rate Avg': 'Fwd Blk Rate Avg', 'fwd_blk_rate_avg': 'Fwd Blk Rate Avg', 'Bwd Avg Bytes/Bulk': 'Bwd Byts/b Avg', 'Bwd Bytes/Bulk Avg': 'Bwd Byts/b Avg', 'bwd_byts_b_avg': 'Bwd Byts/b Avg', 'Bwd Avg Packets/Bulk': 'Bwd Pkts/b Avg', 'Bwd Packet/Bulk Avg': 'Bwd Pkts/b Avg', 'bwd_pkts_b_avg': 'Bwd Pkts/b Avg', 'Bwd Avg Bulk Rate': 'Bwd Blk Rate Avg', 'Bwd Bulk Rate Avg': 'Bwd Blk Rate Avg', 'bwd_blk_rate_avg': 'Bwd Blk Rate Avg', 'Subflow Fwd Packets': 'Subflow Fwd Pkts', 'subflow_fwd_pkts': 'Subflow Fwd Pkts', 'Subflow Fwd Bytes': 'Subflow Fwd Byts', 'subflow_fwd_byts': 'Subflow Fwd Byts', 'Subflow Bwd Packets': 'Subflow Bwd Pkts', 'subflow_bwd_pkts': 'Subflow Bwd Pkts', 'Subflow Bwd Bytes': 'Subflow Bwd Byts', 'subflow_bwd_byts': 'Subflow Bwd Byts', 'Init_Win_bytes_forward': 'Init Fwd Win Byts', 'FWD Init Win Bytes': 'Init Fwd Win Byts', 'init_fwd_win_byts': 'Init Fwd Win Byts', 'Init_Win_bytes_backward': 'Init Bwd Win Byts', 'Bwd Init Win Bytes': 'Init Bwd Win Byts', 'init_bwd_win_byts': 'Init Bwd Win Byts', 'act_data_pkt_fwd': 'Fwd Act Data Pkts', 'Fwd Act Data Pkts': 'Fwd Act Data Pkts', 'fwd_act_data_pkts': 'Fwd Act Data Pkts', 'min_seg_size_forward': 'Fwd Seg Size Min', 'fwd_seg_size_min': 'Fwd Seg Size Min', 'active_mean': 'Active Mean', 'active_std': 'Active Std', 'active_max': 'Active Max', 'active_min': 'Active Min', 'idle_mean': 'Idle Mean', 'idle_std': 'Idle Std', 'idle_max': 'Idle Max', 'idle_min': 'Idle Min', 'Fwd Segment Size Avg': 'Fwd Seg Size Avg', 'Bwd Segment Size Avg': 'Bwd Seg Size Avg'}
LABEL_COLUMN = 'label'
BINARY_COLUMN = 'binary_label'
SOURCE_COLUMN = 'source'
ORIGINAL_LABEL_COLUMN = 'original_label'
BENIGN_TOKENS = {'benign', 'normal', 'normal traffic', 'non attack', 'non-attack', '0'}

def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value)
    text = text.replace('\ufeff', '')
    text = text.replace('�', '-')
    text = text.strip().strip('\'"').casefold()
    text = text.rstrip('.')
    text = re.sub('[_/]+', ' ', text)
    text = re.sub('\\s+', ' ', text)
    return text.strip()

def normalize_label(value):
    text = normalize_text(value)
    text = re.sub('(?:\\s*-\\s*|\\s+)attempted\\s*$', '', text).strip()
    compact_text = re.sub('[^a-z0-9]+', '', text)
    if text in BENIGN_TOKENS or text.startswith('benign'):
        return 'benign'
    if 'ddos' in text or 'distributed denial' in text or 'hoic' in text or ('loic' in text):
        return 'dos'
    if text == 'dos' or 'denial of service' in text or 'slowloris' in text or ('slowhttptest' in text) or ('goldeneye' in text) or ('hulk' in text):
        return 'dos'
    if 'portscan' in text or 'port scan' in text or 'reconnaissance' in text or (text == 'recon') or (text == 'probe') or ('nmap' in text) or ('ipsweep' in text) or ('portsweep' in text) or ('satan' in text):
        return 'reconnaissance'
    if 'webattack' in compact_text or ('web' in compact_text and 'bruteforce' in compact_text) or 'sqlinjection' in compact_text or ('xss' in compact_text) or ('crosssite' in compact_text):
        return 'web_attack'
    if 'brute force' in text or 'bruteforce' in compact_text or 'patator' in text or ('password' in text):
        return 'brute_force'
    if 'bot' in text:
        return 'botnet'
    if 'infiltration' in text or 'infilteration' in text:
        return 'infiltration'
    if 'heartbleed' in text:
        return 'heartbleed'
    if 'exploit' in text:
        return 'exploits'
    if 'fuzz' in text:
        return 'fuzzers'
    if text == 'generic' or 'generic attack' in text:
        return 'generic'
    if 'backdoor' in text or 'back door' in text:
        return 'backdoor'
    if 'shellcode' in text:
        return 'shellcode'
    if 'worm' in text:
        return 'worms'
    if text == 'r2l' or 'remote to local' in text:
        return 'r2l'
    if text == 'u2r' or 'user to root' in text:
        return 'u2r'
    if text == 'analysis':
        return 'analysis'
    return 'other_attack'

def normalize_exact_subtype(value):
    text = normalize_text(value)
    text = re.sub('(?:\\s*[-]\\s*|\\s+)attempted\\s*$', '', text)
    return re.sub('\\s+', ' ', text).strip()
_WEB_ATTACK_LABEL_EXAMPLES = ['Web Attack - Brute Force', 'WebAttack-BruteForce', 'Web Attack – XSS', 'WebAttack-SqlInjection', 'Brute Force -Web']
if any((normalize_label(label) != 'web_attack' for label in _WEB_ATTACK_LABEL_EXAMPLES)):
    raise AssertionError('CIC web-attack label normalization failed.')

def normalize_columns(frame):
    result = frame.copy()
    result.columns = result.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip().str.strip('\'"')
    result = result.loc[:, ~result.columns.duplicated(keep='first')].copy()
    return result

def select_label_column(frame, candidates, source):
    """Return the first available label column in the declared order."""
    normalized_lookup = {normalize_text(column): column for column in frame.columns}
    for candidate in candidates:
        column = normalized_lookup.get(normalize_text(candidate))
        if column is not None:
            return column
    raise KeyError(f'No label column found for {source!r}. Candidates={candidates}; available={frame.columns.tolist()}')

def prepare_cic_schema_frame(raw_frame, source, label_candidates, source_file_column=None):
    frame = normalize_columns(raw_frame)
    frame = frame.rename(columns={**CICFLOWMETER_TO_CIC2017, **UNSW_TO_CIC2017})
    label_column = select_label_column(frame, label_candidates, source=source)
    metadata = pd.DataFrame(index=frame.index)
    metadata[ORIGINAL_LABEL_COLUMN] = frame[label_column].astype('string').str.strip()
    metadata[SOURCE_COLUMN] = source
    if source_file_column is not None and source_file_column in frame.columns:
        metadata['source_file'] = frame[source_file_column].astype('string')
    if 'Fwd Header Length.1' not in frame.columns and 'Fwd Header Length' in frame.columns:
        frame['Fwd Header Length.1'] = frame['Fwd Header Length']
    missing = [column for column in CIC2017_FEATURE_COLUMNS if column not in frame.columns]
    if missing:
        raise KeyError(f'{source} is missing CIC-IDS2017 features: {missing}')
    features = frame[CIC2017_FEATURE_COLUMNS].copy()
    for column in CIC2017_FEATURE_COLUMNS:
        features[column] = pd.to_numeric(features[column], errors='coerce')
    features = features.replace([np.inf, -np.inf], np.nan)
    result = pd.concat([features.reset_index(drop=True), metadata.reset_index(drop=True)], axis=1)
    result[LABEL_COLUMN] = result[ORIGINAL_LABEL_COLUMN].map(normalize_label)
    result[BINARY_COLUMN] = result[LABEL_COLUMN].ne('benign').astype(np.int8)
    return result

def load_cicflow_pool():
    darpa_raw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/CIC-features/Darpa/Darpa_Pcap_CIC_features.csv', low_memory=False)
    unsw_raw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/CIC-features/UNSW-NB15/UNSW_Pcap_CIC_features.csv', low_memory=False)
    cic17_raw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/CIC-features/CIC2017/CIC17_Pcap_CIC_features.csv', low_memory=False)
    cic18_raw = pd.read_csv('/content/drive/MyDrive/Array_submission/Dataset/CIC-features/CIC2018/CIC18_Pcap_CIC_features.csv', low_memory=False)
    darpa = prepare_cic_schema_frame(darpa_raw, source='DARPA', label_candidates=['Attack_category', 'attack_category', 'Label', 'label'])
    unsw = prepare_cic_schema_frame(unsw_raw, source='UNSW-NB15', label_candidates=['category', 'attack_cat', 'Label', 'label'])
    cic17 = prepare_cic_schema_frame(cic17_raw, source='CIC-IDS2017', label_candidates=['Attack_category', 'attack_category', 'category', 'attack_cat', 'Label', 'label'])
    cic18 = prepare_cic_schema_frame(cic18_raw, source='CSE-CIC-IDS2018', label_candidates=['Attack_category', 'attack_category', 'category', 'attack_cat', 'Label', 'label'])
    frame = pd.concat([darpa, unsw, cic17, cic18], ignore_index=True)
    return frame.loc[~frame[LABEL_COLUMN].eq('heartbleed')].reset_index(drop=True)

def cic17_to_cic18_schema(frame):
    renamed = frame.rename(columns={column: CIC_TO_CSE2018_ALIASES[column] for column in frame.columns if column in CIC_TO_CSE2018_ALIASES})
    if renamed.columns.duplicated().any():
        output = pd.DataFrame(index=renamed.index)
        for column in pd.unique(renamed.columns):
            matching = renamed.loc[:, renamed.columns == column]
            output[column] = matching.iloc[:, 0] if matching.shape[1] == 1 else matching.bfill(axis=1).iloc[:, 0]
        renamed = output
    missing = [column for column in CSE2018_FEATURE_COLUMNS if column not in renamed.columns]
    if missing:
        raise KeyError(f'CICFlowMeter partition is missing CSE-CIC-IDS2018 features: {missing}')
    metadata = [ORIGINAL_LABEL_COLUMN, SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN]
    result = renamed[CSE2018_FEATURE_COLUMNS + metadata].copy()
    for column in CSE2018_FEATURE_COLUMNS:
        result[column] = pd.to_numeric(result[column], errors='coerce')
    return result.replace([np.inf, -np.inf], np.nan)


In [6]:
def add_partition_identity(frame, source_column, label_column):
    working = frame.reset_index(drop=True).copy()
    working['__partition_row_id'] = working[source_column].astype(str) + ':' + working.groupby(source_column).cumcount().astype(str)
    working['__binary_stratum'] = np.where(working[label_column].eq('benign'), 'benign', 'malicious')
    return working

def measure_available_flows(frame, representation, source_column, label_column, sources):
    working = add_partition_identity(frame, source_column, label_column)
    rows = []
    for source_name in sources:
        source_frame = working.loc[working[source_column].eq(source_name)]
        if source_frame.empty:
            raise ValueError(f'{representation}: source {source_name!r} is absent.')
        benign_rows = int(source_frame['__binary_stratum'].eq('benign').sum())
        malicious_rows = int(source_frame['__binary_stratum'].eq('malicious').sum())
        if benign_rows == 0 or malicious_rows == 0:
            raise ValueError(f'{representation}/{source_name}: both benign and malicious flows are required; benign={benign_rows}, malicious={malicious_rows}.')
        rows.append({'representation': representation, 'source': source_name, 'available_rows': len(source_frame), 'available_benign': benign_rows, 'available_malicious': malicious_rows})
    category_availability = working.groupby([source_column, label_column]).size().rename('available_rows').reset_index()
    category_availability.insert(0, 'representation', representation)
    return (pd.DataFrame(rows), category_availability)

def balance_complete_pool(frame, representation, source_column, label_column, sources, balance_target, seed=PARTITION_SEED):
    working = add_partition_identity(frame, source_column, label_column)
    selected_parts = []
    audit_rows = []
    for source_offset, source_name in enumerate(sources):
        source_frame = working.loc[working[source_column].eq(source_name)]
        benign_candidates = source_frame.loc[source_frame['__binary_stratum'].eq('benign')]
        malicious_candidates = source_frame.loc[source_frame['__binary_stratum'].eq('malicious')]
        if len(benign_candidates) < balance_target or len(malicious_candidates) < balance_target:
            raise ValueError(f'{representation}/{source_name}: cannot select n_bal={balance_target}; benign={len(benign_candidates)}, malicious={len(malicious_candidates)}.')
        selected_benign = benign_candidates.sample(n=balance_target, replace=False, random_state=seed + 1000 + 2 * source_offset)
        selected_malicious = malicious_candidates.sample(n=balance_target, replace=False, random_state=seed + 1001 + 2 * source_offset)
        selected_parts.extend([selected_benign, selected_malicious])
        audit_rows.append({'representation': representation, 'source': source_name, 'class': 'benign', 'available_before_balancing': len(benign_candidates), 'selected_before_split': balance_target})
        audit_rows.append({'representation': representation, 'source': source_name, 'class': 'malicious', 'available_before_balancing': len(malicious_candidates), 'selected_before_split': balance_target})
    balanced_pool = pd.concat(selected_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    expected_rows = 2 * len(sources) * balance_target
    observed = balanced_pool.groupby([source_column, '__binary_stratum']).size()
    if len(balanced_pool) != expected_rows or len(observed) != 2 * len(sources) or (not observed.eq(balance_target).all()):
        raise AssertionError(f'{representation}: pre-split source/class balancing failed; expected_rows={expected_rows}, observed={observed.to_dict()}')
    return (balanced_pool, pd.DataFrame(audit_rows))

def split_balanced_pool(balanced_pool, representation, source_column, sources, balance_target, seed=PARTITION_SEED, test_size=TEST_SIZE):
    train_parts = []
    test_parts = []
    manifest_rows = []
    expected_test_per_class = math.ceil(balance_target * test_size)
    expected_train_per_class = balance_target - expected_test_per_class
    for source_offset, source_name in enumerate(sources):
        source_frame = balanced_pool.loc[balanced_pool[source_column].eq(source_name)].copy()
        train_part, test_part = train_test_split(source_frame, test_size=test_size, random_state=seed + 2000 + source_offset, shuffle=True, stratify=source_frame['__binary_stratum'])
        train_counts = train_part['__binary_stratum'].value_counts()
        test_counts = test_part['__binary_stratum'].value_counts()
        if int(train_counts.get('benign', 0)) != expected_train_per_class or int(train_counts.get('malicious', 0)) != expected_train_per_class:
            raise AssertionError(f'{representation}/{source_name}: unexpected training counts after 70:30 split; expected={expected_train_per_class}, observed={train_counts.to_dict()}')
        if int(test_counts.get('benign', 0)) != expected_test_per_class or int(test_counts.get('malicious', 0)) != expected_test_per_class:
            raise AssertionError(f'{representation}/{source_name}: unexpected test counts after 70:30 split; expected={expected_test_per_class}, observed={test_counts.to_dict()}')
        train_parts.append(train_part)
        test_parts.append(test_part)
        manifest_rows.append({'representation': representation, 'source': source_name, 'balanced_rows_before_split': len(source_frame), 'balanced_benign_before_split': balance_target, 'balanced_malicious_before_split': balance_target, 'train_rows': len(train_part), 'train_benign': int(train_counts['benign']), 'train_malicious': int(train_counts['malicious']), 'test_rows': len(test_part), 'test_benign': int(test_counts['benign']), 'test_malicious': int(test_counts['malicious']), 'train_pct': 100 * len(train_part) / len(source_frame), 'test_pct': 100 * len(test_part) / len(source_frame)})
    train = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=seed + 3000).reset_index(drop=True)
    test = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=seed + 4000).reset_index(drop=True)
    train_ids = set(train['__partition_row_id'])
    test_ids = set(test['__partition_row_id'])
    balanced_ids = set(balanced_pool['__partition_row_id'])
    if train_ids & test_ids or train_ids | test_ids != balanced_ids:
        raise AssertionError(f'{representation}: training and test partitions are not disjoint and exhaustive relative to the balanced pool.')
    return (train, test, pd.DataFrame(manifest_rows), expected_train_per_class, expected_test_per_class)

def remove_partition_helpers(frame):
    return frame.drop(columns=['__partition_row_id', '__binary_stratum'], errors='ignore')

def save_partition(directory_name, train_frame, test_frame, balanced_pool, source_column, label_column, availability, category_availability, balancing_audit, split_manifest):
    output_dir = PARTITION_ROOT / directory_name
    output_dir.mkdir(parents=True, exist_ok=True)
    train_output = remove_partition_helpers(train_frame)
    test_output = remove_partition_helpers(test_frame)
    balanced_output = remove_partition_helpers(balanced_pool)
    train_output.to_csv(output_dir / 'balanced_train.csv', index=False)
    test_output.to_csv(output_dir / 'test_all_sources.csv', index=False)
    for source_name, source_frame in test_output.groupby(source_column, sort=True):
        safe_name = re.sub('[^A-Za-z0-9_-]+', '_', str(source_name))
        source_frame.to_csv(output_dir / f'test_{safe_name}.csv', index=False)
    balanced_distribution = balanced_output.groupby([source_column, label_column]).size().rename('rows').reset_index()
    training_distribution = train_output.groupby([source_column, label_column]).size().rename('rows').reset_index()
    test_distribution = test_output.groupby([source_column, label_column]).size().rename('rows').reset_index()
    balanced_distribution.to_csv(output_dir / 'balanced_pool_distribution_before_split.csv', index=False)
    training_distribution.to_csv(output_dir / 'balanced_training_distribution.csv', index=False)
    test_distribution.to_csv(output_dir / 'test_distribution.csv', index=False)
    availability.to_csv(output_dir / 'available_binary_distribution.csv', index=False)
    category_availability.to_csv(output_dir / 'available_category_distribution.csv', index=False)
    balancing_audit.to_csv(output_dir / 'balancing_audit.csv', index=False)
    split_manifest.to_csv(output_dir / 'partition_manifest.csv', index=False)
    print(directory_name, 'train=', train_output.shape, 'test=', test_output.shape, 'directory=', output_dir)
    return training_distribution


In [ ]:
representation_specs = [('kdd_nsl', load_kdd_nsl_pool, 'source_pcap', 'label', KDD_SOURCES), ('unsw_nb15', load_unsw_pool, 'source_pcap', 'label', KDD_SOURCES), ('cicflowmeter', load_cicflow_pool, SOURCE_COLUMN, LABEL_COLUMN, CIC_SOURCES)]
availability_parts = []
for representation, loader, source_column, label_column, sources in representation_specs:
    pool = loader()
    availability, category_availability = measure_available_flows(pool, representation, source_column, label_column, sources)
    availability_parts.append(availability)
    del pool, availability, category_availability
    gc.collect()
global_availability = pd.concat(availability_parts, ignore_index=True)
balance_target = int(global_availability['available_malicious'].min())
limiting_combinations = global_availability.loc[global_availability['available_malicious'].eq(balance_target)].reset_index(drop=True)
expected_test_per_class = math.ceil(balance_target * TEST_SIZE)
expected_train_per_class = balance_target - expected_test_per_class
global_availability.to_csv(PARTITION_ROOT / 'global_available_flow_counts.csv', index=False)
pd.DataFrame([{'equation': 'min over representation-source available malicious flows', 'n_bal_before_split': balance_target, 'train_fraction': 1 - TEST_SIZE, 'test_fraction': TEST_SIZE, 'training_flows_per_class_source': expected_train_per_class, 'test_flows_per_class_source': expected_test_per_class, 'training_rows_per_representation': 8 * expected_train_per_class, 'test_rows_per_representation': 8 * expected_test_per_class}]).to_csv(PARTITION_ROOT / 'balance_and_split_target.csv', index=False)
print('Equation-derived pre-split n_bal:', balance_target)
print('Expected per source after split: train=', expected_train_per_class, 'benign +', expected_train_per_class, 'malicious; test=', expected_test_per_class, 'benign +', expected_test_per_class, 'malicious')
training_distribution_parts = []
test_distribution_parts = []
for representation, loader, source_column, label_column, sources in representation_specs:
    pool = loader()
    availability, category_availability = measure_available_flows(pool, representation, source_column, label_column, sources)
    balanced_pool, balancing_audit = balance_complete_pool(pool, representation, source_column, label_column, sources, balance_target)
    train_partition, test_partition, split_manifest, observed_train_per_class, observed_test_per_class = split_balanced_pool(balanced_pool, representation, source_column, sources, balance_target)
    if observed_train_per_class != expected_train_per_class or observed_test_per_class != expected_test_per_class:
        raise AssertionError(f'{representation}: derived split sizes changed unexpectedly.')
    if representation == 'cicflowmeter':
        training_distribution = save_partition('cicflow_cic17', train_partition, test_partition, balanced_pool, source_column, label_column, availability, category_availability, balancing_audit, split_manifest)
        cic18_train = cic17_to_cic18_schema(remove_partition_helpers(train_partition))
        cic18_test = cic17_to_cic18_schema(remove_partition_helpers(test_partition))
        cic18_balanced = cic17_to_cic18_schema(remove_partition_helpers(balanced_pool))
        save_partition('cicflow_cic18', cic18_train, cic18_test, cic18_balanced, source_column, label_column, availability, category_availability, balancing_audit, split_manifest)
    else:
        training_distribution = save_partition(representation, train_partition, test_partition, balanced_pool, source_column, label_column, availability, category_availability, balancing_audit, split_manifest)
    training_distribution = training_distribution.rename(columns={source_column: 'source', label_column: 'category'})
    training_distribution.insert(0, 'representation', representation)
    training_distribution_parts.append(training_distribution)
    test_distribution = remove_partition_helpers(test_partition).groupby([source_column, label_column]).size().rename('rows').reset_index().rename(columns={source_column: 'source', label_column: 'category'})
    test_distribution.insert(0, 'representation', representation)
    test_distribution_parts.append(test_distribution)
    del pool, availability, category_availability, balanced_pool, balancing_audit, train_partition, test_partition, split_manifest, training_distribution, test_distribution
    gc.collect()
combined_training_distribution = pd.concat(training_distribution_parts, ignore_index=True)
combined_test_distribution = pd.concat(test_distribution_parts, ignore_index=True)
combined_training_distribution.to_csv(PARTITION_ROOT / 'training_distribution_by_representation.csv', index=False)
combined_test_distribution.to_csv(PARTITION_ROOT / 'test_distribution_by_representation.csv', index=False)


Equation-derived pre-split n_bal: 11629
Expected per source after split: train= 8140 benign + 8140 malicious; test= 3489 benign + 3489 malicious


kdd_nsl train= (65120, 47) test= (27912, 47) directory= /content/drive/MyDrive/Array_submission/Prepared_partitions/splits/kdd_nsl
unsw_nb15 train= (65120, 53) test= (27912, 53) directory= /content/drive/MyDrive/Array_submission/Prepared_partitions/splits/unsw_nb15
cicflow_cic17 train= (65120, 82) test= (27912, 82) directory= /content/drive/MyDrive/Array_submission/Prepared_partitions/splits/cicflow_cic17
cicflow_cic18 train= (65120, 81) test= (27912, 81) directory= /content/drive/MyDrive/Array_submission/Prepared_partitions/splits/cicflow_cic18
